In [0]:
from pyspark.pipelines import materialized_view, table
from pyspark.sql import functions as F


# =============================================================================
# Materialized View: customers (Batch Query)
# =============================================================================

@materialized_view(
    comment="Customer dimension table from Unity Catalog volume CSV",
    table_properties={"delta.logRetentionDuration": "interval 30 days"}
)
def customers():
    """
    Batch materialized view for customer dimension data.

    Source: Unity Catalog volume (CSV file)
    Location: /Volumes/main/default/landing/customers.csv
    Refresh: On-demand or scheduled

    Schema:
        customer_id: int (PK)
        name: string
        email: string
        region: string
        signup_date: date

    Data Quality:
        - Filter out NULL names (basic data quality)
    """
    return (spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load("/Volumes/main/default/landing/customers.csv")
            .filter(F.col("name").isNotNull())
            .withColumn("signup_date", F.col("signup_date").cast("date")))